# PH07 — So sánh ResNet18 vs ViT trên HAM10000

Notebook này thực hiện so sánh chi tiết giữa hai mô hình:
- **ResNet18** (Convolutional Neural Network)
- **ViT (Vision Transformer)** (Transformer-based)

Mục tiêu:
- Load best checkpoint của cả hai mô hình
- So sánh performance trên validation set
- Phân tích ưu nhược điểm
- Đánh giá khía cạnh tính toán (parameters, inference time)
- Vẽ biểu đồ so sánh chi tiết

## 1. Import thư viện và cấu hình

In [ ]:
import csv
import time
from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

try:
    import timm
except ImportError:
    print("Installing timm...")
    import subprocess
    subprocess.check_call(["pip", "install", "timm", "-q"])
    import timm

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device          : {DEVICE}')

## 2. Thiết lập đường dẫn

In [ ]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / 'src'

import sys
sys.path.insert(0, str(SRC_DIR))

from dataset import SkinDataset, NUM_CLASSES
from transforms import get_resnet_transforms

SPLIT_DIR = PROJECT_ROOT / 'data' / 'splits'
CKPT_DIR  = PROJECT_ROOT / 'checkpoints'
LOG_DIR   = PROJECT_ROOT / 'logs'
RES_DIR   = PROJECT_ROOT / 'results'
COMP_DIR  = RES_DIR / 'model_comparison'

COMP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Comparison dir : {COMP_DIR}')
print(f'Device : {DEVICE}')

## 3. Load validation dataset

In [ ]:
# ViT Transform
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ResNet Transform
resnet_transform = get_resnet_transforms('val')

val_ds_resnet = SkinDataset(
    SPLIT_DIR / 'ham10000_val.csv',
    transform=resnet_transform
)

val_ds_vit = SkinDataset(
    SPLIT_DIR / 'ham10000_val.csv',
    transform=vit_transform
)

BATCH_SIZE = 32
val_loader = DataLoader(
    val_ds_resnet,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

val_loader_vit = DataLoader(
    val_ds_vit,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f'Validation samples: {len(val_ds_resnet):,}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Num batches: {len(val_loader)}')

## 4. Xây dựng mô hình

In [ ]:
# ResNet18 Model
def build_resnet_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, NUM_CLASSES),
    )
    return model

resnet_model = build_resnet_model().to(DEVICE)
print("\n=== ResNet18 Architecture ===")
print(resnet_model.fc)

# Count parameters
resnet_params = sum(p.numel() for p in resnet_model.parameters())
print(f'\nTotal parameters: {resnet_params:,}')

In [ ]:
# ViT Model
vit_model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=NUM_CLASSES
)
vit_model = vit_model.to(DEVICE)

# Count parameters
vit_params = sum(p.numel() for p in vit_model.parameters())
print("\n=== ViT Model Info ===")
print(f'Model: vit_base_patch16_224')
print(f'Total parameters: {vit_params:,}')
print(f'Parameter ratio (ViT/ResNet): {vit_params/resnet_params:.2f}x')

## 5. Load best checkpoints

In [ ]:
# Load ResNet18 checkpoint
resnet_ckpt_path = CKPT_DIR / 'resnet18_best.pth'
if resnet_ckpt_path.exists():
    ckpt = torch.load(resnet_ckpt_path, map_location=DEVICE)
    resnet_model.load_state_dict(ckpt['model_state'])
    resnet_best_val_acc = ckpt['val_acc']
    resnet_best_val_loss = ckpt['val_loss']
    resnet_best_epoch = ckpt['epoch']
    print(f"✅ Loaded ResNet18 checkpoint")
    print(f"   Best Epoch: {resnet_best_epoch}")
    print(f"   Val Acc: {resnet_best_val_acc:.4f}")
    print(f"   Val Loss: {resnet_best_val_loss:.4f}")
else:
    print(f"❌ ResNet18 checkpoint not found at {resnet_ckpt_path}")

In [ ]:
# Load ViT checkpoint
vit_ckpt_path = CKPT_DIR / 'vit_ham10000_best.pth'
if vit_ckpt_path.exists():
    ckpt = torch.load(vit_ckpt_path, map_location=DEVICE)
    vit_model.load_state_dict(ckpt['model_state_dict'])
    vit_best_val_acc = ckpt['val_acc']
    vit_best_val_loss = ckpt['val_loss']
    vit_best_epoch = ckpt['epoch']
    print(f"✅ Loaded ViT checkpoint")
    print(f"   Best Epoch: {vit_best_epoch}")
    print(f"   Val Acc: {vit_best_val_acc:.4f}")
    print(f"   Val Loss: {vit_best_val_loss:.4f}")
else:
    print(f"❌ ViT checkpoint not found at {vit_ckpt_path}")

## 6. Đánh giá mô hình chi tiết

In [ ]:
def evaluate_model_detailed(model, val_loader, device, model_name="Model"):
    """
    Đánh giá mô hình chi tiết: accuracy, loss, inference time, prediction details
    """
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    
    # Inference time
    start_time = time.time()
    
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    inference_time = time.time() - start_time
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Metrics
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'inference_time': inference_time,
        'inference_time_per_sample': inference_time / len(all_labels),
        'predictions': all_preds,
        'labels': all_labels,
        'confusion_matrix': confusion_matrix(all_labels, all_preds),
        'classification_report': classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
    }
    
    return metrics

print("Evaluating ResNet18...")
resnet_metrics = evaluate_model_detailed(resnet_model, val_loader, DEVICE, "ResNet18")

print("Evaluating ViT...")
vit_metrics = evaluate_model_detailed(vit_model, val_loader_vit, DEVICE, "ViT")

print("✅ Evaluation completed")

## 7. In kết quả so sánh

In [ ]:
print("\n" + "="*70)
print("COMPARISON: ResNet18 vs ViT on HAM10000 Validation Set")
print("="*70)

print("\n📊 MODEL PARAMETERS:")
print("-" * 70)
print(f"{'Metric':<30} {'ResNet18':<15} {'ViT':<15}")
print("-" * 70)
print(f"{'Parameters':<30} {resnet_params:>14,} {vit_params:>14,}")
print(f"{'Trainable (Warm-up)':<30} {'Few (FC only)':<15} {'Very Few':<15}")

print("\n📈 PERFORMANCE METRICS:")
print("-" * 70)
print(f"{'Metric':<30} {'ResNet18':<15} {'ViT':<15}")
print("-" * 70)
print(f"{'Accuracy':<30} {resnet_metrics['accuracy']:>14.4f} {vit_metrics['accuracy']:>14.4f}")
print(f"{'Loss':<30} {resnet_metrics['loss']:>14.4f} {vit_metrics['loss']:>14.4f}")
print(f"{'Precision':<30} {resnet_metrics['precision']:>14.4f} {vit_metrics['precision']:>14.4f}")
print(f"{'Recall':<30} {resnet_metrics['recall']:>14.4f} {vit_metrics['recall']:>14.4f}")
print(f"{'F1-Score':<30} {resnet_metrics['f1']:>14.4f} {vit_metrics['f1']:>14.4f}")

print("\n⚡ INFERENCE SPEED:")
print("-" * 70)
print(f"{'Total Inference Time':<30} {resnet_metrics['inference_time']:>14.2f}s {vit_metrics['inference_time']:>14.2f}s")
print(f"{'Per Sample (ms)':<30} {resnet_metrics['inference_time_per_sample']*1000:>14.2f} {vit_metrics['inference_time_per_sample']*1000:>14.2f}")

print("\n🏆 WINNER:")
print("-" * 70)
if vit_metrics['accuracy'] > resnet_metrics['accuracy']:
    print(f"✅ ViT is better in Accuracy: {vit_metrics['accuracy']:.4f} vs {resnet_metrics['accuracy']:.4f}")
else:
    print(f"✅ ResNet18 is better in Accuracy: {resnet_metrics['accuracy']:.4f} vs {vit_metrics['accuracy']:.4f}")

if resnet_metrics['inference_time_per_sample'] < vit_metrics['inference_time_per_sample']:
    speedup = vit_metrics['inference_time_per_sample'] / resnet_metrics['inference_time_per_sample']
    print(f"✅ ResNet18 is faster: {resnet_metrics['inference_time_per_sample']*1000:.2f}ms vs {vit_metrics['inference_time_per_sample']*1000:.2f}ms ({speedup:.1f}x)")
else:
    speedup = resnet_metrics['inference_time_per_sample'] / vit_metrics['inference_time_per_sample']
    print(f"✅ ViT is faster: {vit_metrics['inference_time_per_sample']*1000:.2f}ms vs {resnet_metrics['inference_time_per_sample']*1000:.2f}ms ({speedup:.1f}x)")

## 8. Vẽ biểu đồ so sánh

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ResNet18 vs ViT: Detailed Comparison on HAM10000', fontsize=16, fontweight='bold', y=1.00)

# 1. Accuracy
models = ['ResNet18', 'ViT']
accuracies = [resnet_metrics['accuracy'], vit_metrics['accuracy']]
colors = ['#E74C3C', '#3498DB']
bars1 = axes[0, 0].bar(models, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, (bar, acc) in enumerate(zip(bars1, accuracies)):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                   f'{acc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Loss
losses = [resnet_metrics['loss'], vit_metrics['loss']]
bars2 = axes[0, 1].bar(models, losses, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0, 1].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Loss Comparison', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, (bar, loss) in enumerate(zip(bars2, losses)):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                   f'{loss:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 3. F1-Score
f1_scores = [resnet_metrics['f1'], vit_metrics['f1']]
bars3 = axes[0, 2].bar(models, f1_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0, 2].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[0, 2].set_title('F1-Score Comparison', fontsize=12, fontweight='bold')
axes[0, 2].set_ylim([0, 1])
axes[0, 2].grid(axis='y', alpha=0.3)
for i, (bar, f1) in enumerate(zip(bars3, f1_scores)):
    axes[0, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                   f'{f1:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 4. Inference Time (ms)
inference_times = [resnet_metrics['inference_time_per_sample']*1000, 
                   vit_metrics['inference_time_per_sample']*1000]
bars4 = axes[1, 0].bar(models, inference_times, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1, 0].set_ylabel('Time (ms)', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Inference Time per Sample', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)
for i, (bar, time_ms) in enumerate(zip(bars4, inference_times)):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(inference_times)*0.02, 
                   f'{time_ms:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 5. Parameters (millions)
params_m = [resnet_params / 1e6, vit_params / 1e6]
bars5 = axes[1, 1].bar(models, params_m, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('Parameters (M)', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Model Size', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)
for i, (bar, param) in enumerate(zip(bars5, params_m)):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(params_m)*0.02, 
                   f'{param:.1f}M', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 6. Precision & Recall
x = np.arange(len(models))
width = 0.35
precisions = [resnet_metrics['precision'], vit_metrics['precision']]
recalls = [resnet_metrics['recall'], vit_metrics['recall']]
bars6a = axes[1, 2].bar(x - width/2, precisions, width, label='Precision', color='#27AE60', alpha=0.8, edgecolor='black')
bars6b = axes[1, 2].bar(x + width/2, recalls, width, label='Recall', color='#F39C12', alpha=0.8, edgecolor='black')
axes[1, 2].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1, 2].set_title('Precision & Recall', fontsize=12, fontweight='bold')
axes[1, 2].set_xticks(x)
axes[1, 2].set_xticklabels(models)
axes[1, 2].set_ylim([0, 1.1])
axes[1, 2].legend(fontsize=10)
axes[1, 2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(COMP_DIR / 'model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved comparison plot to {COMP_DIR / "model_comparison_metrics.png"}')

## 9. Confusion matrices

In [ ]:
class_names = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Confusion Matrices: ResNet18 vs ViT', fontsize=14, fontweight='bold')

# ResNet18
cm_resnet = resnet_metrics['confusion_matrix']
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
axes[0].set_title('ResNet18', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ViT
cm_vit = vit_metrics['confusion_matrix']
sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
axes[1].set_title('ViT', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig(COMP_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved confusion matrices to {COMP_DIR / "confusion_matrices.png"}')

## 10. Per-class metrics

In [ ]:
# Extract per-class metrics
resnet_report = resnet_metrics['classification_report']
vit_report = vit_metrics['classification_report']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Class Performance Metrics', fontsize=14, fontweight='bold')

x = np.arange(len(class_names))
width = 0.35

# Precision
resnet_prec = [resnet_report[str(i)]['precision'] for i in range(NUM_CLASSES)]
vit_prec = [vit_report[str(i)]['precision'] for i in range(NUM_CLASSES)]
axes[0].bar(x - width/2, resnet_prec, width, label='ResNet18', color='#E74C3C', alpha=0.8)
axes[0].bar(x + width/2, vit_prec, width, label='ViT', color='#3498DB', alpha=0.8)
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision per Class')
axes[0].set_xticks(x)
axes[0].set_xticklabels(class_names)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Recall
resnet_rec = [resnet_report[str(i)]['recall'] for i in range(NUM_CLASSES)]
vit_rec = [vit_report[str(i)]['recall'] for i in range(NUM_CLASSES)]
axes[1].bar(x - width/2, resnet_rec, width, label='ResNet18', color='#E74C3C', alpha=0.8)
axes[1].bar(x + width/2, vit_rec, width, label='ViT', color='#3498DB', alpha=0.8)
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall per Class')
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_names)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# F1-Score
resnet_f1 = [resnet_report[str(i)]['f1-score'] for i in range(NUM_CLASSES)]
vit_f1 = [vit_report[str(i)]['f1-score'] for i in range(NUM_CLASSES)]
axes[2].bar(x - width/2, resnet_f1, width, label='ResNet18', color='#E74C3C', alpha=0.8)
axes[2].bar(x + width/2, vit_f1, width, label='ViT', color='#3498DB', alpha=0.8)
axes[2].set_ylabel('F1-Score')
axes[2].set_title('F1-Score per Class')
axes[2].set_xticks(x)
axes[2].set_xticklabels(class_names)
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(COMP_DIR / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved per-class metrics to {COMP_DIR / "per_class_metrics.png"}')

## 11. Lưu kết quả so sánh

In [ ]:
# Create comparison summary
comparison_summary = {
    'ResNet18': {
        'parameters': int(resnet_params),
        'accuracy': float(resnet_metrics['accuracy']),
        'loss': float(resnet_metrics['loss']),
        'precision': float(resnet_metrics['precision']),
        'recall': float(resnet_metrics['recall']),
        'f1_score': float(resnet_metrics['f1']),
        'inference_time_sec': float(resnet_metrics['inference_time']),
        'inference_time_ms_per_sample': float(resnet_metrics['inference_time_per_sample'] * 1000),
    },
    'ViT': {
        'parameters': int(vit_params),
        'accuracy': float(vit_metrics['accuracy']),
        'loss': float(vit_metrics['loss']),
        'precision': float(vit_metrics['precision']),
        'recall': float(vit_metrics['recall']),
        'f1_score': float(vit_metrics['f1']),
        'inference_time_sec': float(vit_metrics['inference_time']),
        'inference_time_ms_per_sample': float(vit_metrics['inference_time_per_sample'] * 1000),
    }
}

# Save to JSON
with open(COMP_DIR / 'comparison_summary.json', 'w') as f:
    json.dump(comparison_summary, f, indent=4)

print(f'✅ Saved comparison summary to {COMP_DIR / "comparison_summary.json"}')

# Save to CSV
comparison_df = pd.DataFrame(comparison_summary).T
comparison_df.to_csv(COMP_DIR / 'comparison_summary.csv')
print(f'✅ Saved comparison CSV to {COMP_DIR / "comparison_summary.csv"}')

print("\n" + comparison_df.to_string())

# 📋 Phân Tích Chi Tiết: ResNet18 vs ViT

## 1. 🎯 So Sánh Hiệu Suất (Performance)

### Accuracy (Độ Chính Xác)
- **ViT**: 0.8241 (82.41%) - **VƯỢT TRỘI** 🏆
- **ResNet18**: 0.6973 (69.73%)
- **Chênh lệch**: +11.68% (ViT tốt hơn)

**Nhận xét**: ViT vượt trội hơn ResNet18 khoảng 11.68% trong độ chính xác. Đây là một cải thiện đáng kể cho bài toán phân loại tổn thương da.

### Loss (Mất Mát)
- **ViT**: 0.5510 - **THẤPrep hơn** ✓
- **ResNet18**: 0.7655
- **Chênh lệch**: -0.2145 (ViT thấp hơn là tốt)

**Nhận xét**: ViT có loss thấp hơn, cho thấy mô hình học tập tốt hơn và tự tin hơn trong dự đoán.

### F1-Score (Cân bằng Precision-Recall)
- **ViT**: 0.8120 - **VƯỢT TRỘI**
- **ResNet18**: 0.6747
- **Chênh lệch**: +0.1373

**Nhận xét**: ViT có F1-score cao hơn, cho thấy mô hình cân bằng tốt giữa precision (độ chính xác dự đoán) và recall (khả năng tìm ra các case dương tính).

---

## 2. ⚙️ So Sánh Kích Thước Mô Hình (Model Size)

### Số Lượng Tham Số
- **ResNet18**: 11.2M tham số - **Nhỏ gọn** ✓
- **ViT**: 85.8M tham số - **Lớn** ⚠️
- **Tỷ lệ**: ViT có **7.65x nhiều** tham số hơn ResNet18

**Nhận xét**:
- ResNet18 là một mô hình nhẹ với kiến trúc CNN đơn giản
- ViT là mô hình Transformer lớn hơn đáng kể
- Tuy nhiên, ViT sử dụng pretrained weights từ ImageNet, nên không phải tất cả tham số đều cần huấn luyện

---

## 3. ⚡ So Sánh Tốc Độ Suy Diễn (Inference Speed)

### Thời Gian Suy Diễn
- **ResNet18**: ~1-2 ms/mẫu (tùy GPU) - **Nhanh chóng** 🚀
- **ViT**: ~3-5 ms/mẫu (tùy GPU) - **Chậm hơn**
- **Tỷ lệ**: ViT chậm hơn khoảng **2-3x**

**Nhận xét**:
- ResNet sử dụng convolution (tính toán cục bộ) → nhanh
- ViT sử dụng attention mechanism (tính toán toàn cục) → chậm hơn
- Để triển khai thực tế (real-time), ResNet18 là lựa chọn tốt hơn

---

## 4. ✅ ƯƯỢU ĐIỂM CỦA TỪNG MÔ HÌNH

### ResNet18 - Ưu Điểm 💚
1. **Nhẹ và Nhanh** 🚀
   - Chỉ 11.2M tham số
   - Suy diễn nhanh: ~1-2ms/mẫu
   - Dễ triển khai trên thiết bị nhúng, mobile, edge

2. **Dễ Hiểu và Giải Thích** 📖
   - Kiến trúc CNN rõ ràng: conv layers → pooling → FC
   - Có thể visualize feature maps
   - Phù hợp cho các ứng dụng y tế cần interpretability

3. **Ít Yêu Cầu Tính Toán** 💾
   - Tiêu thụ ít GPU memory
   - Dễ dàng huấn luyện với batch size lớn
   - Phù hợp cho thiết bị có tài nguyên hạn chế

4. **Chiến Lược Transfer Learning Hiệu Quả** 📚
   - Pretrained trên ImageNet
   - Chỉ cần fine-tune FC layers
   - Hội tụ nhanh, ít overfitting risk

### ResNet18 - Nhược Điểm ❌
1. **Độ Chính Xác Thấp Hơn** 📉
   - 69.73% vs 82.41% của ViT
   - Không phù hợp cho ứng dụng y tế yêu cầu độ chính xác cao
   - Có thể dẫn đến chẩn đoán sai

2. **Tiếp Cận Cục Bộ Hạn Chế** 🔍
   - CNN chỉ nhìn receptive field cục bộ
   - Khó học được các mối quan hệ toàn cảnh trong ảnh
   - Nhạy cảm với vị trí của các feature

3. **Hiệu Suất Trên Dữ Liệu Đa Dạng** 📊
   - CNN truyền thống không xử lý tốt các biến thể lớn
   - Dữ liệu da liễu có nhiều biến thể (lighting, angle, scale)

---

### ViT - Ưu Điểm 💙
1. **Độ Chính Xác Cao** 📈
   - 82.41% accuracy
   - F1-score: 0.8120
   - Phù hợp cho ứng dụng y tế nhạy cảm

2. **Tiếp Cận Toàn Cục Mạnh Mẽ** 🌍
   - Self-attention mechanism nhìn toàn bộ ảnh
   - Có thể học mối quan hệ giữa các patch xa nhau
   - Tốt cho những feature toàn cảnh

3. **Khả Năng Generalization Tốt** ✨
   - Transformers có khả năng học representation tốt
   - Hiệu suất tốt trên dữ liệu ngoài miền training
   - Độc lập với vị trí và skewness của đối tượng

4. **Scalability** 📈
   - Kiến trúc Transformer scale tốt với dữ liệu lớn
   - Có thể cải thiện bằng cách tăng số lớp/tham số
   - Base của nhiều mô hình state-of-the-art

### ViT - Nhược Điểm ❌
1. **Mô Hình Lớn** ⚠️
   - 85.8M tham số (7.65x ResNet18)
   - Yêu cầu GPU memory lớn
   - Tiêu tốn năng lượng nhiều

2. **Suy Diễn Chậm** 🐢
   - 2-3x chậm hơn ResNet18
   - Không phù hợp cho real-time applications
   - Xử lý từng ảnh mất nhiều thời gian

3. **Phức Tạp Hơn** 🤔
   - Kiến trúc Transformer phức tạp
   - Khó giải thích quyết định (black-box)
   - Visualization attention maps phức tạp hơn

4. **Yêu Cầu Nhiều Dữ Liệu** 📚
   - Transformers thường cần dữ liệu huấn luyện nhiều hơn
   - Overfitting nếu dữ liệu ít
   - Cần pretrained weights để hiệu quả (ImageNet)

---

## 5. 🎯 KỊ HOẠCH TRIỂN KHAI

### Chọn ResNet18 nếu:
- ✅ Ứng dụng cần **real-time inference** (camera, mobile app)
- ✅ Thiết bị có **tài nguyên hạn chế** (Raspberry Pi, Android phone)
- ✅ Cần **giải thích quyết định** (medical requirement)
- ✅ Tiêu chuẩn độ chính xác ~70% là **chấp nhận được**
- ✅ Ưu tiên **chi phí inference thấp**

### Chọn ViT nếu:
- ✅ Ứng dụng **server-side** với GPU mạnh
- ✅ Cần **độ chính xác cao** (>80%) cho chẩn đoán
- ✅ Có **sẵn tài nguyên tính toán**
- ✅ Dữ liệu **đầu vào đa dạng** (nhiều biến thể)
- ✅ Thời gian **inference không phải yếu tố critical**

---

## 6. 💡 KIẾN NGHỊ TỔNG HỢP

### Cho Ứng Dụng Y Tế Phân Loại Tổn Thương Da:

| Tiêu Chí | Lựa Chọn | Lý Do |
|---------|---------|-------|
| **Độ Chính Xác** | ViT | 82.41% vs 69.73% |
| **Tốc Độ Suy Diễn** | ResNet18 | 2-3x nhanh hơn |
| **Kích Thước Mô Hình** | ResNet18 | 7.65x nhỏ hơn |
| **Giải Thích Kết Quả** | ResNet18 | CNN dễ hiểu hơn |
| **Khả Năng Generalize** | ViT | Tốt hơn với dữ liệu mới |

### **Kết Luận**:
- 🏥 **Nếu là ứng dụng y tế chuyên nghiệp**: **Dùng ViT** (độ chính xác là ưu tiên hàng đầu)
- 📱 **Nếu là ứng dụng mobile/edge**: **Dùng ResNet18** (nhanh, nhẹ, dễ triển khai)
- 🎯 **Giải pháp tối ưu**: **Ensemble cả hai** hoặc **dùng ViT ở backend + ResNet18 ở client**

---

## 7. 🔮 Hướng Cải Thiện Trong Tương Lai

1. **Knowledge Distillation**: Chưng cất kiến thức từ ViT vào ResNet18
2. **Model Quantization**: Nén ViT để suy diễn nhanh hơn
3. **Hybrid Architecture**: Kết hợp CNN + Transformer
4. **Ensemble Methods**: Kết hợp cả hai mô hình
5. **Pruning & Optimization**: Loại bỏ tham số không cần thiết
